In [1]:
import math
import itertools
from typing import Dict, Any, List, Tuple

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# =========================================================
# UPDATED SOIL DATABASE
# =========================================================
soil_db = pd.DataFrame([
    {
        "soil_type": "CG",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 0.0,
        "phi_intermediate": 36.0, "phi_lowc_highphi": 40.0, "phi_highc_lowphi": 32.0,
        "bearing_Low": 392.4, "bearing_Medium": 588.6, "bearing_High": 784.8
    },
    {
        "soil_type": "CG PF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 29.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "CG NPF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 30.0, "phi_lowc_highphi": 32.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "Silt",
        "gamma_bulk": 18.5, "gamma_sub": 8.69,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 25.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 20.0,
        "bearing_Low": 49.05, "bearing_Medium": 147.15, "bearing_High": 294.3
    },
    {
        "soil_type": "CL",
        "gamma_bulk": 17.5, "gamma_sub": 7.69,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 18.0, "phi_lowc_highphi": 27.0, "phi_highc_lowphi": 15.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "CH",
        "gamma_bulk": 19.0, "gamma_sub": 9.19,
        "c_intermediate": 20.0, "c_lowc_highphi": 10.0, "c_highc_lowphi": 40.0,
        "phi_intermediate": 15.0, "phi_lowc_highphi": 20.0, "phi_highc_lowphi": 12.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "Boulder",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 60.0, "phi_lowc_highphi": 60.0, "phi_highc_lowphi": 37.0,
        "bearing_Low": 981.0, "bearing_Medium": 981.0, "bearing_High": 981.0
    },
])

DENSITY_OPTIONS = ["Low", "Medium", "High"]
STRENGTH_STATE_OPTIONS = ["Intermediate", "Low C High Phi", "High C Low Phi"]
OLD_PA_SOILS = {"CG", "CG NPF", "Boulder"}
DEFAULT_OUTPUT_FILE = "retaining_wall_fos_results.xlsx"

REQUIRED_FOS_OVERTURNING = 2.0
REQUIRED_FOS_SLIDING = 1.5
REQUIRED_FOS_BEARING = 3.0

ANGLE_RATIO_FRONT_FACE = 30.0  # 1:30


# =========================================================
# SAFE INPUT HELPERS
# =========================================================
def ask_positive_int(prompt: str) -> int:
    while True:
        raw = input(prompt).strip()
        try:
            value = int(raw)
            if value <= 0:
                print("Please enter a positive whole number.")
                continue
            return value
        except ValueError:
            print("Invalid input. Please enter a whole number like 1, 2, 3.")


def normalize_strength_state(text: str) -> str:
    t = " ".join(text.strip().split()).lower()
    mapping = {
        "intermediate": "Intermediate",
        "low c high phi": "Low C High Phi",
        "high c low phi": "High C Low Phi",
    }
    return mapping.get(t, text.title())


def ask_values(label: str, example: str, cast_func=float, allow_text=False, allowed_values=None):
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        n = ask_positive_int("How many values? : ")

        raw = input("What are they? (comma separated) : ").strip()
        if raw == "":
            print("Input cannot be empty. Please try again.")
            continue

        parts = [x.strip() for x in raw.split(",") if x.strip()]

        if len(parts) != n:
            print(f"You said {n} values, but entered {len(parts)} values. Please enter again.")
            continue

        try:
            if allow_text:
                values = [str(x).strip() for x in parts]

                if allowed_values is not None:
                    normalized = []
                    for v in values:
                        if allowed_values == STRENGTH_STATE_OPTIONS:
                            vv = normalize_strength_state(v)
                        else:
                            vv = v.title()
                        normalized.append(vv)

                    bad = [v for v in normalized if v not in allowed_values]
                    if bad:
                        print(f"These values are not allowed: {bad}")
                        print(f"Allowed values are: {allowed_values}")
                        continue
                    return normalized

                return values
            else:
                return [cast_func(x) for x in parts]

        except ValueError:
            print("Invalid input. Please enter again.")


def ask_yes_no(label: str, example: str = "Yes") -> bool:
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        value = input("Enter Yes or No: ").strip().lower()

        if value in {"yes", "y", "true", "1"}:
            return True
        elif value in {"no", "n", "false", "0"}:
            return False
        else:
            print("Invalid input. Please enter Yes or No.")


def ask_soil_types(available_soils):
    while True:
        values = ask_values(
            label="1. Soil types",
            example="CG, CG PF, Silt, CL",
            cast_func=str,
            allow_text=True
        )

        bad = [v for v in values if v not in available_soils]
        if bad:
            print(f"These soil types are not in the database: {bad}")
            print(f"Available soil types are: {available_soils}")
            continue
        return values


def ask_output_filename(default_name: str) -> str:
    while True:
        value = input(f"\n14. Output Excel file name [default: {default_name}] : ").strip()
        if value == "":
            return default_name
        if not value.lower().endswith(".xlsx"):
            value += ".xlsx"
        return value


# =========================================================
# SOIL LOOKUP
# =========================================================
def get_phi_c_by_strength_state(row: Dict[str, Any], strength_state: str) -> Tuple[float, float]:
    if strength_state == "Intermediate":
        return row["phi_intermediate"], row["c_intermediate"]
    elif strength_state == "Low C High Phi":
        return row["phi_lowc_highphi"], row["c_lowc_highphi"]
    elif strength_state == "High C Low Phi":
        return row["phi_highc_lowphi"], row["c_highc_lowphi"]
    else:
        raise ValueError(f"Unknown strength state: {strength_state}")


def soil_lookup(soil_type: str, density_type: str, strength_state: str) -> Dict[str, Any]:
    density_type = density_type.title()
    if density_type not in DENSITY_OPTIONS:
        raise ValueError(f"density_type must be one of {DENSITY_OPTIONS}")

    if strength_state not in STRENGTH_STATE_OPTIONS:
        raise ValueError(f"strength_state must be one of {STRENGTH_STATE_OPTIONS}")

    row = soil_db.loc[soil_db["soil_type"] == soil_type]
    if row.empty:
        raise ValueError(f"Soil type '{soil_type}' not found.")

    row = row.iloc[0].to_dict()
    phi, cohesion = get_phi_c_by_strength_state(row, strength_state)

    row["density_type"] = density_type
    row["strength_state"] = strength_state
    row["phi_used"] = phi
    row["cohesion_used"] = cohesion
    row["bearing_capacity"] = row.get(f"bearing_{density_type}")
    return row


# =========================================================
# GEOTECHNICAL FUNCTIONS
# =========================================================
def rankine_Ka_Kp_sloping(beta_deg: float, phi_deg: float) -> Tuple[float, float]:
    beta = math.radians(beta_deg)
    phi = math.radians(phi_deg)

    cosb = math.cos(beta)
    term_sq = cosb**2 - math.cos(phi)**2

    if term_sq < 0:
        raise ValueError(f"Invalid case: beta={beta_deg}° is greater than allowable for phi={phi_deg}°.")

    term = math.sqrt(term_sq)

    Ka = cosb * (cosb - term) / (cosb + term)
    Kp = cosb * (cosb + term) / (cosb - term)
    return Ka, Kp


def active_pressure_old(H: float, gamma: float, c: float, Ka: float) -> float:
    Pa = 0.5 * Ka * gamma * H**2 - 2.0 * c * math.sqrt(Ka) * H
    return max(Pa, 0.0)


def active_pressure_new(H: float, gamma: float, c: float, Ka: float) -> Tuple[float, float]:
    if gamma <= 0 or Ka <= 0:
        return 0.0, 0.0

    Zc = (2.0 * c) / (gamma * math.sqrt(Ka))

    if H <= Zc:
        return 0.0, Zc

    Pa = 0.5 * (H - Zc) * (gamma * H * Ka - 2.0 * c * math.sqrt(Ka))
    return max(Pa, 0.0), Zc


def compute_case(
    soil_type: str,
    density_type: str,
    strength_state: str,
    H: float,
    beta_deg: float,
    BC_coef: float,
    DE_coef: float,
    EF_coef: float,
    DB_coef: float,
    h_toe: float,
    top_width: float,
    gamma_concrete: float = 24.0,
) -> Dict[str, Any]:

    soil = soil_lookup(soil_type, density_type, strength_state)

    gamma_bulk = soil["gamma_bulk"]
    gamma_sub = soil["gamma_sub"]
    phi = soil["phi_used"]
    cohesion = soil["cohesion_used"]
    bearing = soil["bearing_capacity"]

    delta_deg = (2.0 / 3.0) * phi

    BC = BC_coef * H
    DE = DE_coef * H
    EF_input = EF_coef * H
    DB = DB_coef * H

    angle_displacement = (H - DB) / ANGLE_RATIO_FRONT_FACE
    min_ef_required = top_width + angle_displacement

    # UPDATED RULE: if EF <= top width, always use corrected EF
    if EF_input <= top_width:
        EF_used = min_ef_required
        ef_rule_applied = "YES"
    else:
        EF_used = EF_input
        ef_rule_applied = "NO"

    ef_check = "PASS" if EF_used >= min_ef_required else "FAIL"

    FG = BC - (DE + EF_used)

    if FG <= 0:
        raise ValueError("FG <= 0. Increase BC/H or reduce DE/H and EF/H.")

    beta_rad = math.radians(beta_deg)
    H_prime = H + math.tan(beta_rad) * FG

    Ka, Kp = rankine_Ka_Kp_sloping(beta_deg, phi)

    # Weights with corrected EF_used
    W1 = 0.5 * (H_prime - H) * FG * gamma_bulk
    W2 = FG * (H - DB) * gamma_bulk
    W3 = top_width * (H - DB) * gamma_concrete
    W4 = 0.5 * (H - DB) * max(EF_used - top_width, 0.0) * gamma_concrete
    W5 = DB * BC * gamma_concrete
    W_total = W1 + W2 + W3 + W4 + W5

    if soil_type in OLD_PA_SOILS:
        Pa = active_pressure_old(H, gamma_bulk, cohesion, Ka)
        Zc = 0.0
        active_formula_used = "Old formula"
    else:
        Pa, Zc = active_pressure_new(H, gamma_bulk, cohesion, Ka)
        active_formula_used = "New formula with Zc"

    Pp = 0.5 * Kp * gamma_bulk * (h_toe ** 2) + 2.0 * cohesion * math.sqrt(Kp) * h_toe

    Ph = Pa * math.cos(beta_rad)
    Pv = Pa * math.sin(beta_rad)

    # Moments with corrected EF_used
    x_W1 = DE + EF_used + 2.0 * FG / 3.0
    x_W2 = DE + EF_used + FG / 2.0
    x_W3 = DE + max(EF_used - top_width, 0.0) + top_width / 2.0
    x_W4 = DE + 2.0 * max(EF_used - top_width, 0.0) / 3.0 if W4 > 0 else 0.0
    x_W5 = BC / 2.0

    M_W1 = W1 * x_W1
    M_W2 = W2 * x_W2
    M_W3 = W3 * x_W3
    M_W4 = W4 * x_W4
    M_W5 = W5 * x_W5
    M_Pv = Pv * BC

    Mr = M_W1 + M_W2 + M_W3 + M_W4 + M_W5 + M_Pv
    Mo = Ph * H_prime / 3.0

    FS_overturning = Mr / Mo if abs(Mo) > 1e-12 else float("inf")

    delta_rad = math.radians(delta_deg)
    resisting = W_total * math.tan(delta_rad) + Pp
    driving = Ph
    FS_sliding = resisting / driving if abs(driving) > 1e-12 else float("inf")

    V = W_total + Pv
    e = (BC / 2.0) - ((Mr - Mo) / V) if abs(V) > 1e-12 else float("nan")
    pmax = (V / BC) * (1.0 + 6.0 * e / BC) if BC > 0 else float("nan")
    pmin = (V / BC) * (1.0 - 6.0 * e / BC) if BC > 0 else float("nan")

    FS_bearing = "N/A"
    if bearing is not None and not pd.isna(bearing) and max(pmax, pmin) > 0:
        FS_bearing = bearing / max(pmax, pmin)

    overturning_check = "PASS" if FS_overturning >= REQUIRED_FOS_OVERTURNING else "FAIL"
    sliding_check = "PASS" if FS_sliding >= REQUIRED_FOS_SLIDING else "FAIL"
    if FS_bearing == "N/A":
        bearing_check = "N/A"
    else:
        bearing_check = "PASS" if FS_bearing >= REQUIRED_FOS_BEARING else "FAIL"

    return {
        "Soil types": soil_type,
        "Density type": density_type,
        "Strength state": strength_state,
        "Height of structure": H,
        "inclination of soil": beta_deg,
        "Coefficient of length (BC/H)": BC_coef,
        "Coefficient of base (DE/H)": DE_coef,
        "Coefficient of EF (EF/H)": EF_coef,
        "Coefficient of height of toe (DB/H)": DB_coef,
        "Toe soil height h": h_toe,
        "Top width": top_width,

        "Length BC": BC,
        "Length DE": DE,
        "Length EF input": EF_input,
        "Length EF used": EF_used,
        "Length FG": FG,
        "Height DB": DB,

        "Angle ratio used": f"1:{int(ANGLE_RATIO_FRONT_FACE)}",
        "Angle displacement": angle_displacement,
        "Minimum EF required": min_ef_required,
        "EF rule applied": ef_rule_applied,
        "EF Check": ef_check,

        "Bulk unit weight": gamma_bulk,
        "Submerged unit weight": gamma_sub,
        "phi used": phi,
        "cohesion used": cohesion,
        "Bearing capacity": bearing,
        "delta": delta_deg,
        "Ka": Ka,
        "Kp": Kp,
        "Zc": Zc,
        "Active formula used": active_formula_used,

        "W1": W1,
        "W2": W2,
        "W3": W3,
        "W4": W4,
        "W5": W5,
        "W_total": W_total,
        "Pa": Pa,
        "Pp": Pp,
        "Ph": Ph,
        "Pv": Pv,

        "x_W1": x_W1,
        "x_W2": x_W2,
        "x_W3": x_W3,
        "x_W4": x_W4,
        "x_W5": x_W5,
        "M_W1": M_W1,
        "M_W2": M_W2,
        "M_W3": M_W3,
        "M_W4": M_W4,
        "M_W5": M_W5,
        "M_Pv": M_Pv,
        "Mr": Mr,
        "Mo": Mo,

        "eccentricity": e,
        "pmax": pmax,
        "pmin": pmin,

        "FOS_against Overturning": FS_overturning,
        "FOS_against Sliding": FS_sliding,
        "FOS_against Bearing": FS_bearing,
        "Required FOS Overturning": REQUIRED_FOS_OVERTURNING,
        "Required FOS Sliding": REQUIRED_FOS_SLIDING,
        "Required FOS Bearing": REQUIRED_FOS_BEARING,
        "Overturning Check": overturning_check,
        "Sliding Check": sliding_check,
        "Bearing Check": bearing_check,
    }


# =========================================================
# DATAFRAME BUILDERS
# =========================================================
def build_summary_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    cols = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length BC",
        "Length DE",
        "Length EF input",
        "Length EF used",
        "Length FG",
        "Height DB",
        "Angle ratio used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "phi used",
        "cohesion used",
        "Bearing capacity",
        "Ka",
        "Kp",
        "Zc",
        "Active formula used",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "x_W1",
        "x_W2",
        "x_W3",
        "x_W4",
        "x_W5",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "pmax",
        "pmin",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
        "Required FOS Overturning",
        "Required FOS Sliding",
        "Required FOS Bearing",
        "Overturning Check",
        "Sliding Check",
        "Bearing Check",
    ]
    return pd.DataFrame(results)[cols]


def build_detailed_2col_sheet(results: List[Dict[str, Any]]) -> pd.DataFrame:
    wanted_order = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length EF input",
        "Length EF used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
    ]

    blocks = []
    for i, res in enumerate(results, start=1):
        block = pd.DataFrame({
            f"Parameter_{i}": wanted_order,
            f"Value_{i}": [res.get(k, "") for k in wanted_order]
        })
        blocks.append(block)

    return pd.concat(blocks, axis=1)


def autosize_excel_columns(writer, sheet_name, df):
    worksheet = writer.sheets[sheet_name]
    for idx, col in enumerate(df.columns):
        series = df[col].fillna("").astype(str) if col in df.columns else pd.Series(dtype=str)
        max_len = max([len(str(col))] + [len(x) for x in series.tolist()] + [10])
        worksheet.set_column(idx, idx, min(max_len + 2, 18 if sheet_name == "Summary" else 45))


# =========================================================
# MAIN
# =========================================================
def main():
    print("=" * 74)
    print("RETAINING WALL COMBINATION ANALYSIS")
    print("=" * 74)
    print("This script will ask the user for all values, run all combinations,")
    print("calculate FoS against overturning, sliding, and bearing,")
    print("and export the results to Excel.\n")

    print("Available soil types:")
    print(", ".join(soil_db["soil_type"].tolist()))
    print("\nAvailable density types:")
    print("Low, Medium, High")
    print("Example input: Medium, High")
    print("\nAvailable strength states:")
    print("Intermediate, Low C High Phi, High C Low Phi")
    print("Example input: Intermediate, High C Low Phi")
    print("\nDE hint:")
    print("Suggested DE range: 0.13H to 0.23H")
    print(f"\nEF rule:")
    print(f"If EF <= top width, use EF = top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)}")

    soil_types = ask_soil_types(soil_db["soil_type"].tolist())

    H_values = ask_values(
        label="2. Height of structure H (m)",
        example="2.5, 5, 7",
        cast_func=float
    )

    beta_values = ask_values(
        label="3. Inclination of soil β (degree)",
        example="0, 5, 10, 15",
        cast_func=float
    )

    BC_coef_values = ask_values(
        label="4. Coefficient of length BC/H",
        example="0.5, 0.6, 0.7",
        cast_func=float
    )

    DE_coef_values = ask_values(
        label="5. Coefficient of base DE/H",
        example="0.13, 0.18, 0.23",
        cast_func=float
    )

    EF_coef_values = ask_values(
        label="6. Coefficient of EF (EF/H)",
        example="0.1, 0.15, 0.2",
        cast_func=float
    )

    DB_coef_values = ask_values(
        label="7. Coefficient of height of toe DB/H",
        example="0.05, 0.10, 0.15",
        cast_func=float
    )

    density_types = ask_values(
        label="8. Soil density type(s)",
        example="Medium, High",
        cast_func=str,
        allow_text=True,
        allowed_values=DENSITY_OPTIONS
    )

    strength_states = ask_values(
        label="9. Soil strength state(s)",
        example="Intermediate, High C Low Phi",
        cast_func=str,
        allow_text=True,
        allowed_values=STRENGTH_STATE_OPTIONS
    )

    h_toe_values = ask_values(
        label="10. Toe soil height h (m)",
        example="0.6, 0.7",
        cast_func=float
    )

    top_width_values = ask_values(
        label="11. Top width values (m)",
        example="0.3, 0.4, 0.5",
        cast_func=float
    )

    use_default_gamma_conc = ask_yes_no(
        label="12. Use default unit weight of concrete = 24 kN/m3 ?",
        example="Yes"
    )
    if use_default_gamma_conc:
        gamma_concrete = 24.0
    else:
        while True:
            try:
                gamma_concrete = float(input("Enter concrete unit weight (kN/m3): ").strip())
                if gamma_concrete <= 0:
                    print("Unit weight must be positive.")
                    continue
                break
            except ValueError:
                print("Invalid input. Please enter a number.")

    output_file = ask_output_filename(DEFAULT_OUTPUT_FILE)

    combos = list(itertools.product(
        soil_types,
        density_types,
        strength_states,
        H_values,
        beta_values,
        BC_coef_values,
        DE_coef_values,
        EF_coef_values,
        DB_coef_values,
        h_toe_values,
        top_width_values
    ))

    print(f"\nTotal combinations to evaluate: {len(combos)}")

    results = []
    errors = []

    for idx, (soil_type, density_type, strength_state, H, beta, BCc, DEc, EFc, DBc, htoe, topw) in enumerate(combos, start=1):
        try:
            res = compute_case(
                soil_type=soil_type,
                density_type=density_type,
                strength_state=strength_state,
                H=H,
                beta_deg=beta,
                BC_coef=BCc,
                DE_coef=DEc,
                EF_coef=EFc,
                DB_coef=DBc,
                h_toe=htoe,
                top_width=topw,
                gamma_concrete=gamma_concrete
            )
            results.append(res)
        except Exception as e:
            errors.append({
                "Combination No.": idx,
                "Soil": soil_type,
                "Density": density_type,
                "Strength State": strength_state,
                "H": H,
                "beta": beta,
                "BC/H": BCc,
                "DE/H": DEc,
                "EF/H": EFc,
                "DB/H": DBc,
                "Toe soil height h": htoe,
                "Top width": topw,
                "Error": str(e)
            })

    if not results:
        print("\nNo valid combinations were solved.")
        if errors:
            display(pd.DataFrame(errors))
        return

    summary_df = build_summary_dataframe(results)
    detailed_df = build_detailed_2col_sheet(results)
    errors_df = pd.DataFrame(errors)

    inputs_info = pd.DataFrame({
        "Input item": [
            "Soil types",
            "Height values H",
            "Beta values",
            "BC/H values",
            "DE/H values",
            "EF/H values",
            "DB/H values",
            "Toe soil height h values",
            "Top width values",
            "Density types",
            "Strength states",
            "Concrete unit weight",
            "Old active pressure soils",
            "Active pressure rule for other soils",
            "Required FOS Overturning",
            "Required FOS Sliding",
            "Required FOS Bearing",
            "Bearing selection rule",
            "DE hint",
            "EF correction rule"
        ],
        "Value": [
            ", ".join(soil_types),
            ", ".join(map(str, H_values)),
            ", ".join(map(str, beta_values)),
            ", ".join(map(str, BC_coef_values)),
            ", ".join(map(str, DE_coef_values)),
            ", ".join(map(str, EF_coef_values)),
            ", ".join(map(str, DB_coef_values)),
            ", ".join(map(str, h_toe_values)),
            ", ".join(map(str, top_width_values)),
            ", ".join(density_types),
            ", ".join(strength_states),
            gamma_concrete,
            ", ".join(sorted(OLD_PA_SOILS)),
            "Use new formula with Zc",
            REQUIRED_FOS_OVERTURNING,
            REQUIRED_FOS_SLIDING,
            REQUIRED_FOS_BEARING,
            "Low density uses bearing_Low, Medium uses bearing_Medium, High uses bearing_High",
            "Suggested DE range = 0.13H to 0.23H",
            f"If EF <= top width, use EF = top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)}"
        ]
    })

    soil_properties_used = soil_db.copy()

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        inputs_info.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=0)
        soil_properties_used.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=len(inputs_info) + 3)

        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        detailed_df.to_excel(writer, sheet_name="Detailed_2Col", index=False)

        if not errors_df.empty:
            errors_df.to_excel(writer, sheet_name="Errors", index=False)

        ws_inputs = writer.sheets["Inputs_Used"]
        ws_inputs.set_column(0, 0, 32)
        ws_inputs.set_column(1, 1, 60)
        ws_inputs.freeze_panes(1, 0)

        autosize_excel_columns(writer, "Summary", summary_df)
        autosize_excel_columns(writer, "Detailed_2Col", detailed_df)

        if not errors_df.empty:
            autosize_excel_columns(writer, "Errors", errors_df)
            writer.sheets["Errors"].freeze_panes(1, 0)

        writer.sheets["Summary"].freeze_panes(1, 0)
        writer.sheets["Detailed_2Col"].freeze_panes(1, 0)

    print("\nDone.")
    print(f"Valid combinations solved: {len(results)}")
    print(f"Failed combinations: {len(errors)}")
    print(f"Excel file created: {output_file}")

    print("\nPreview of Summary sheet:")
    display(summary_df.head())


# =========================================================
# RUN
# =========================================================
main()


RETAINING WALL COMBINATION ANALYSIS
This script will ask the user for all values, run all combinations,
calculate FoS against overturning, sliding, and bearing,
and export the results to Excel.

Available soil types:
CG, CG PF, CG NPF, Silt, CL, CH, Boulder

Available density types:
Low, Medium, High
Example input: Medium, High

Available strength states:
Intermediate, Low C High Phi, High C Low Phi
Example input: Intermediate, High C Low Phi

DE hint:
Suggested DE range: 0.13H to 0.23H

EF rule:
If EF <= top width, use EF = top width + (H-DB)/30

1. Soil types
Example: CG, CG PF, Silt, CL


How many values? :  1
What are they? (comma separated) :  CG



2. Height of structure H (m)
Example: 2.5, 5, 7


How many values? :  5
What are they? (comma separated) :  2, 3, 5, 7, 9



3. Inclination of soil β (degree)
Example: 0, 5, 10, 15


...

How many values? :  2
What are they? (comma separated) :  .3, .4



12. Use default unit weight of concrete = 24 kN/m3 ?
Example: Yes


Enter Yes or No:  Yes

14. Output Excel file name [default: retaining_wall_fos_results.xlsx] :  CG final 1



Total combinations to evaluate: 14400


ValueError: This sheet is too large! Your sheet size is: 39, 28680 Max sheet size is: 1048576, 16384

## Another code updated

In [7]:
import math
import itertools
from typing import Dict, Any, List, Tuple

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# =========================================================
# UPDATED SOIL DATABASE
# =========================================================
soil_db = pd.DataFrame([
    {
        "soil_type": "CG",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 0.0,
        "phi_intermediate": 36.0, "phi_lowc_highphi": 40.0, "phi_highc_lowphi": 32.0,
        "bearing_Low": 392.4, "bearing_Medium": 588.6, "bearing_High": 784.8
    },
    {
        "soil_type": "CG PF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 29.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "CG NPF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 30.0, "phi_lowc_highphi": 32.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "Silt",
        "gamma_bulk": 18.5, "gamma_sub": 8.69,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 25.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 20.0,
        "bearing_Low": 49.05, "bearing_Medium": 147.15, "bearing_High": 294.3
    },
    {
        "soil_type": "CL",
        "gamma_bulk": 17.5, "gamma_sub": 7.69,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 18.0, "phi_lowc_highphi": 27.0, "phi_highc_lowphi": 15.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "CH",
        "gamma_bulk": 19.0, "gamma_sub": 9.19,
        "c_intermediate": 20.0, "c_lowc_highphi": 10.0, "c_highc_lowphi": 40.0,
        "phi_intermediate": 15.0, "phi_lowc_highphi": 20.0, "phi_highc_lowphi": 12.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "Boulder",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 60.0, "phi_lowc_highphi": 60.0, "phi_highc_lowphi": 37.0,
        "bearing_Low": 981.0, "bearing_Medium": 981.0, "bearing_High": 981.0
    },
])

DENSITY_OPTIONS = ["Low", "Medium", "High"]
STRENGTH_STATE_OPTIONS = ["Intermediate", "Low C High Phi", "High C Low Phi"]
OLD_PA_SOILS = {"CG", "CG NPF", "Boulder"}
DEFAULT_OUTPUT_FILE = "retaining_wall_fos_results.xlsx"

REQUIRED_FOS_OVERTURNING = 2.0
REQUIRED_FOS_SLIDING = 1.5
REQUIRED_FOS_BEARING = 3.0

ANGLE_RATIO_FRONT_FACE = 30.0  # 1:30


# =========================================================
# SAFE INPUT HELPERS
# =========================================================
def ask_positive_int(prompt: str) -> int:
    while True:
        raw = input(prompt).strip()
        try:
            value = int(raw)
            if value <= 0:
                print("Please enter a positive whole number.")
                continue
            return value
        except ValueError:
            print("Invalid input. Please enter a whole number like 1, 2, 3.")


def normalize_strength_state(text: str) -> str:
    t = " ".join(text.strip().split()).lower()
    mapping = {
        "intermediate": "Intermediate",
        "low c high phi": "Low C High Phi",
        "high c low phi": "High C Low Phi",
    }
    return mapping.get(t, text.title())


def ask_values(label: str, example: str, cast_func=float, allow_text=False, allowed_values=None):
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        n = ask_positive_int("How many values? : ")

        raw = input("What are they? (comma separated) : ").strip()
        if raw == "":
            print("Input cannot be empty. Please try again.")
            continue

        parts = [x.strip() for x in raw.split(",") if x.strip()]

        if len(parts) != n:
            print(f"You said {n} values, but entered {len(parts)} values. Please enter again.")
            continue

        try:
            if allow_text:
                values = [str(x).strip() for x in parts]

                if allowed_values is not None:
                    normalized = []
                    for v in values:
                        if allowed_values == STRENGTH_STATE_OPTIONS:
                            vv = normalize_strength_state(v)
                        else:
                            vv = v.title()
                        normalized.append(vv)

                    bad = [v for v in normalized if v not in allowed_values]
                    if bad:
                        print(f"These values are not allowed: {bad}")
                        print(f"Allowed values are: {allowed_values}")
                        continue
                    return normalized

                return values
            else:
                return [cast_func(x) for x in parts]

        except ValueError:
            print("Invalid input. Please enter again.")


def ask_yes_no(label: str, example: str = "Yes") -> bool:
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        value = input("Enter Yes or No: ").strip().lower()

        if value in {"yes", "y", "true", "1"}:
            return True
        elif value in {"no", "n", "false", "0"}:
            return False
        else:
            print("Invalid input. Please enter Yes or No.")


def ask_soil_types(available_soils):
    while True:
        values = ask_values(
            label="1. Soil types",
            example="CG, CG PF, Silt, CL",
            cast_func=str,
            allow_text=True
        )

        bad = [v for v in values if v not in available_soils]
        if bad:
            print(f"These soil types are not in the database: {bad}")
            print(f"Available soil types are: {available_soils}")
            continue
        return values


def ask_output_filename(default_name: str) -> str:
    while True:
        value = input(f"\n14. Output Excel file name [default: {default_name}] : ").strip()
        if value == "":
            return default_name
        if not value.lower().endswith(".xlsx"):
            value += ".xlsx"
        return value


# =========================================================
# SOIL LOOKUP
# =========================================================
def get_phi_c_by_strength_state(row: Dict[str, Any], strength_state: str) -> Tuple[float, float]:
    if strength_state == "Intermediate":
        return row["phi_intermediate"], row["c_intermediate"]
    elif strength_state == "Low C High Phi":
        return row["phi_lowc_highphi"], row["c_lowc_highphi"]
    elif strength_state == "High C Low Phi":
        return row["phi_highc_lowphi"], row["c_highc_lowphi"]
    else:
        raise ValueError(f"Unknown strength state: {strength_state}")


def soil_lookup(soil_type: str, density_type: str, strength_state: str) -> Dict[str, Any]:
    density_type = density_type.title()
    if density_type not in DENSITY_OPTIONS:
        raise ValueError(f"density_type must be one of {DENSITY_OPTIONS}")

    if strength_state not in STRENGTH_STATE_OPTIONS:
        raise ValueError(f"strength_state must be one of {STRENGTH_STATE_OPTIONS}")

    row = soil_db.loc[soil_db["soil_type"] == soil_type]
    if row.empty:
        raise ValueError(f"Soil type '{soil_type}' not found.")

    row = row.iloc[0].to_dict()
    phi, cohesion = get_phi_c_by_strength_state(row, strength_state)

    row["density_type"] = density_type
    row["strength_state"] = strength_state
    row["phi_used"] = phi
    row["cohesion_used"] = cohesion
    row["bearing_capacity"] = row.get(f"bearing_{density_type}")
    return row


# =========================================================
# GEOTECHNICAL FUNCTIONS
# =========================================================
def rankine_Ka_Kp_sloping(beta_deg: float, phi_deg: float) -> Tuple[float, float]:
    beta = math.radians(beta_deg)
    phi = math.radians(phi_deg)

    cosb = math.cos(beta)
    term_sq = cosb**2 - math.cos(phi)**2

    if term_sq < 0:
        raise ValueError(f"Invalid case: beta={beta_deg}° is greater than allowable for phi={phi_deg}°.")

    term = math.sqrt(term_sq)

    Ka = cosb * (cosb - term) / (cosb + term)
    Kp = cosb * (cosb + term) / (cosb - term)
    return Ka, Kp


def active_pressure_old(H: float, gamma: float, c: float, Ka: float) -> float:
    Pa = 0.5 * Ka * gamma * H**2 - 2.0 * c * math.sqrt(Ka) * H
    return max(Pa, 0.0)


def active_pressure_new(H: float, gamma: float, c: float, Ka: float) -> Tuple[float, float]:
    if gamma <= 0 or Ka <= 0:
        return 0.0, 0.0

    Zc = (2.0 * c) / (gamma * math.sqrt(Ka))

    if H <= Zc:
        return 0.0, Zc

    Pa = 0.5 * (H - Zc) * (gamma * H * Ka - 2.0 * c * math.sqrt(Ka))
    return max(Pa, 0.0), Zc


def compute_case(
    BC = BC_coef * H
    DE = DE_coef * H
    EF_input = EF_coef * H
    DB = DB_coef * H

    angle_displacement = (H - DB) / ANGLE_RATIO_FRONT_FACE
    min_ef_required = top_width + angle_displacement

    # EXACT RULE:
    # EF_used = max(EF_input, top_width + (H-DB)/30)
    EF_used = max(EF_input, min_ef_required)

    if EF_used > EF_input:
        ef_rule_applied = "YES"
    else:
        ef_rule_applied = "NO"

    ef_check = "PASS" if EF_used >= min_ef_required else "FAIL"
) -> Dict[str, Any]:

    soil = soil_lookup(soil_type, density_type, strength_state)

    gamma_bulk = soil["gamma_bulk"]
    gamma_sub = soil["gamma_sub"]
    phi = soil["phi_used"]
    cohesion = soil["cohesion_used"]
    bearing = soil["bearing_capacity"]

    delta_deg = (2.0 / 3.0) * phi

    BC = BC_coef * H
    DE = DE_coef * H
    EF_input = EF_coef * H
    DB = DB_coef * H

    angle_displacement = (H - DB) / ANGLE_RATIO_FRONT_FACE
    min_ef_required = top_width + angle_displacement

    # UPDATED RULE: if EF <= top width, always use corrected EF
    if EF_input <= top_width:
        EF_used = min_ef_required
        ef_rule_applied = "YES"
    else:
        EF_used = EF_input
        ef_rule_applied = "NO"

    ef_check = "PASS" if EF_used >= min_ef_required else "FAIL"

    FG = BC - (DE + EF_used)

    if FG <= 0:
        raise ValueError("FG <= 0. Increase BC/H or reduce DE/H and EF/H.")

    beta_rad = math.radians(beta_deg)
    H_prime = H + math.tan(beta_rad) * FG

    Ka, Kp = rankine_Ka_Kp_sloping(beta_deg, phi)

    # Weights with corrected EF_used
    W1 = 0.5 * (H_prime - H) * FG * gamma_bulk
    W2 = FG * (H - DB) * gamma_bulk
    W3 = top_width * (H - DB) * gamma_concrete
    W4 = 0.5 * (H - DB) * max(EF_used - top_width, 0.0) * gamma_concrete
    W5 = DB * BC * gamma_concrete
    W_total = W1 + W2 + W3 + W4 + W5

    if soil_type in OLD_PA_SOILS:
        Pa = active_pressure_old(H, gamma_bulk, cohesion, Ka)
        Zc = 0.0
        active_formula_used = "Old formula"
    else:
        Pa, Zc = active_pressure_new(H, gamma_bulk, cohesion, Ka)
        active_formula_used = "New formula with Zc"

    Pp = 0.5 * Kp * gamma_bulk * (h_toe ** 2) + 2.0 * cohesion * math.sqrt(Kp) * h_toe

    Ph = Pa * math.cos(beta_rad)
    Pv = Pa * math.sin(beta_rad)

    # Moments with corrected EF_used
    x_W1 = DE + EF_used + 2.0 * FG / 3.0
    x_W2 = DE + EF_used + FG / 2.0
    x_W3 = DE + max(EF_used - top_width, 0.0) + top_width / 2.0
    x_W4 = DE + 2.0 * max(EF_used - top_width, 0.0) / 3.0 if W4 > 0 else 0.0
    x_W5 = BC / 2.0

    M_W1 = W1 * x_W1
    M_W2 = W2 * x_W2
    M_W3 = W3 * x_W3
    M_W4 = W4 * x_W4
    M_W5 = W5 * x_W5
    M_Pv = Pv * BC

    Mr = M_W1 + M_W2 + M_W3 + M_W4 + M_W5 + M_Pv
    Mo = Ph * H_prime / 3.0

    FS_overturning = Mr / Mo if abs(Mo) > 1e-12 else float("inf")

    delta_rad = math.radians(delta_deg)
    resisting = W_total * math.tan(delta_rad) + Pp
    driving = Ph
    FS_sliding = resisting / driving if abs(driving) > 1e-12 else float("inf")

    V = W_total + Pv
    e = (BC / 2.0) - ((Mr - Mo) / V) if abs(V) > 1e-12 else float("nan")
    pmax = (V / BC) * (1.0 + 6.0 * e / BC) if BC > 0 else float("nan")
    pmin = (V / BC) * (1.0 - 6.0 * e / BC) if BC > 0 else float("nan")

    FS_bearing = "N/A"
    if bearing is not None and not pd.isna(bearing) and max(pmax, pmin) > 0:
        FS_bearing = bearing / max(pmax, pmin)

    overturning_check = "PASS" if FS_overturning >= REQUIRED_FOS_OVERTURNING else "FAIL"
    sliding_check = "PASS" if FS_sliding >= REQUIRED_FOS_SLIDING else "FAIL"
    if FS_bearing == "N/A":
        bearing_check = "N/A"
    else:
        bearing_check = "PASS" if FS_bearing >= REQUIRED_FOS_BEARING else "FAIL"

    return {
        "Soil types": soil_type,
        "Density type": density_type,
        "Strength state": strength_state,
        "Height of structure": H,
        "inclination of soil": beta_deg,
        "Coefficient of length (BC/H)": BC_coef,
        "Coefficient of base (DE/H)": DE_coef,
        "Coefficient of EF (EF/H)": EF_coef,
        "Coefficient of height of toe (DB/H)": DB_coef,
        "Toe soil height h": h_toe,
        "Top width": top_width,

        "Length BC": BC,
        "Length DE": DE,
        "Length EF input": EF_input,
        "Length EF used": EF_used,
        "Length FG": FG,
        "Height DB": DB,

        "Angle ratio used": f"1:{int(ANGLE_RATIO_FRONT_FACE)}",
        "Angle displacement": angle_displacement,
        "Minimum EF required": min_ef_required,
        "EF rule applied": ef_rule_applied,
        "EF Check": ef_check,

        "Bulk unit weight": gamma_bulk,
        "Submerged unit weight": gamma_sub,
        "phi used": phi,
        "cohesion used": cohesion,
        "Bearing capacity": bearing,
        "delta": delta_deg,
        "Ka": Ka,
        "Kp": Kp,
        "Zc": Zc,
        "Active formula used": active_formula_used,

        "W1": W1,
        "W2": W2,
        "W3": W3,
        "W4": W4,
        "W5": W5,
        "W_total": W_total,
        "Pa": Pa,
        "Pp": Pp,
        "Ph": Ph,
        "Pv": Pv,

        "x_W1": x_W1,
        "x_W2": x_W2,
        "x_W3": x_W3,
        "x_W4": x_W4,
        "x_W5": x_W5,
        "M_W1": M_W1,
        "M_W2": M_W2,
        "M_W3": M_W3,
        "M_W4": M_W4,
        "M_W5": M_W5,
        "M_Pv": M_Pv,
        "Mr": Mr,
        "Mo": Mo,

        "eccentricity": e,
        "pmax": pmax,
        "pmin": pmin,

        "FOS_against Overturning": FS_overturning,
        "FOS_against Sliding": FS_sliding,
        "FOS_against Bearing": FS_bearing,
        "Required FOS Overturning": REQUIRED_FOS_OVERTURNING,
        "Required FOS Sliding": REQUIRED_FOS_SLIDING,
        "Required FOS Bearing": REQUIRED_FOS_BEARING,
        "Overturning Check": overturning_check,
        "Sliding Check": sliding_check,
        "Bearing Check": bearing_check,
    }


# =========================================================
# DATAFRAME BUILDERS
# =========================================================
def build_summary_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    cols = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length BC",
        "Length DE",
        "Length EF input",
        "Length EF used",
        "Length FG",
        "Height DB",
        "Angle ratio used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "phi used",
        "cohesion used",
        "Bearing capacity",
        "Ka",
        "Kp",
        "Zc",
        "Active formula used",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "x_W1",
        "x_W2",
        "x_W3",
        "x_W4",
        "x_W5",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "pmax",
        "pmin",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
        "Required FOS Overturning",
        "Required FOS Sliding",
        "Required FOS Bearing",
        "Overturning Check",
        "Sliding Check",
        "Bearing Check",
    ]
    return pd.DataFrame(results)[cols]


def build_detailed_2col_sheet(results: List[Dict[str, Any]]) -> pd.DataFrame:
    wanted_order = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length EF input",
        "Length EF used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
    ]

    blocks = []
    for i, res in enumerate(results, start=1):
        block = pd.DataFrame({
            f"Parameter_{i}": wanted_order,
            f"Value_{i}": [res.get(k, "") for k in wanted_order]
        })
        blocks.append(block)

    return pd.concat(blocks, axis=1)


def autosize_excel_columns(writer, sheet_name, df):
    worksheet = writer.sheets[sheet_name]
    for idx, col in enumerate(df.columns):
        series = df[col].fillna("").astype(str) if col in df.columns else pd.Series(dtype=str)
        max_len = max([len(str(col))] + [len(x) for x in series.tolist()] + [10])
        worksheet.set_column(idx, idx, min(max_len + 2, 18 if sheet_name == "Summary" else 45))


# =========================================================
# MAIN
# =========================================================
def main():
    print("=" * 74)
    print("RETAINING WALL COMBINATION ANALYSIS")
    print("=" * 74)
    print("This script will ask the user for all values, run all combinations,")
    print("calculate FoS against overturning, sliding, and bearing,")
    print("and export the results to Excel.\n")

    print("Available soil types:")
    print(", ".join(soil_db["soil_type"].tolist()))
    print("\nAvailable density types:")
    print("Low, Medium, High")
    print("Example input: Medium, High")
    print("\nAvailable strength states:")
    print("Intermediate, Low C High Phi, High C Low Phi")
    print("Example input: Intermediate, High C Low Phi")
    print("\nDE hint:")
    print("Suggested DE range: 0.13H to 0.23H")
    print(f"\nEF rule:")
    print(f"If EF <= top width, use EF = top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)}")

    soil_types = ask_soil_types(soil_db["soil_type"].tolist())

    H_values = ask_values(
        label="2. Height of structure H (m)",
        example="2.5, 5, 7",
        cast_func=float
    )

    beta_values = ask_values(
        label="3. Inclination of soil β (degree)",
        example="0, 5, 10, 15",
        cast_func=float
    )

    BC_coef_values = ask_values(
        label="4. Coefficient of length BC/H",
        example="0.5, 0.6, 0.7",
        cast_func=float
    )

    DE_coef_values = ask_values(
        label="5. Coefficient of base DE/H",
        example="0.13, 0.18, 0.23",
        cast_func=float
    )

    EF_coef_values = ask_values(
        label="6. Coefficient of EF (EF/H)",
        example="0.1, 0.15, 0.2",
        cast_func=float
    )

    DB_coef_values = ask_values(
        label="7. Coefficient of height of toe DB/H",
        example="0.05, 0.10, 0.15",
        cast_func=float
    )

    density_types = ask_values(
        label="8. Soil density type(s)",
        example="Medium, High",
        cast_func=str,
        allow_text=True,
        allowed_values=DENSITY_OPTIONS
    )

    strength_states = ask_values(
        label="9. Soil strength state(s)",
        example="Intermediate, High C Low Phi",
        cast_func=str,
        allow_text=True,
        allowed_values=STRENGTH_STATE_OPTIONS
    )

    h_toe_values = ask_values(
        label="10. Toe soil height h (m)",
        example="0.6, 0.7",
        cast_func=float
    )

    top_width_values = ask_values(
        label="11. Top width values (m)",
        example="0.3, 0.4, 0.5",
        cast_func=float
    )

    use_default_gamma_conc = ask_yes_no(
        label="12. Use default unit weight of concrete = 24 kN/m3 ?",
        example="Yes"
    )
    if use_default_gamma_conc:
        gamma_concrete = 24.0
    else:
        while True:
            try:
                gamma_concrete = float(input("Enter concrete unit weight (kN/m3): ").strip())
                if gamma_concrete <= 0:
                    print("Unit weight must be positive.")
                    continue
                break
            except ValueError:
                print("Invalid input. Please enter a number.")

    output_file = ask_output_filename(DEFAULT_OUTPUT_FILE)

    combos = list(itertools.product(
        soil_types,
        density_types,
        strength_states,
        H_values,
        beta_values,
        BC_coef_values,
        DE_coef_values,
        EF_coef_values,
        DB_coef_values,
        h_toe_values,
        top_width_values
    ))

    print(f"\nTotal combinations to evaluate: {len(combos)}")

    results = []
    errors = []

    for idx, (soil_type, density_type, strength_state, H, beta, BCc, DEc, EFc, DBc, htoe, topw) in enumerate(combos, start=1):
        try:
            res = compute_case(
                soil_type=soil_type,
                density_type=density_type,
                strength_state=strength_state,
                H=H,
                beta_deg=beta,
                BC_coef=BCc,
                DE_coef=DEc,
                EF_coef=EFc,
                DB_coef=DBc,
                h_toe=htoe,
                top_width=topw,
                gamma_concrete=gamma_concrete
            )
            results.append(res)
        except Exception as e:
            errors.append({
                "Combination No.": idx,
                "Soil": soil_type,
                "Density": density_type,
                "Strength State": strength_state,
                "H": H,
                "beta": beta,
                "BC/H": BCc,
                "DE/H": DEc,
                "EF/H": EFc,
                "DB/H": DBc,
                "Toe soil height h": htoe,
                "Top width": topw,
                "Error": str(e)
            })

    if not results:
        print("\nNo valid combinations were solved.")
        if errors:
            display(pd.DataFrame(errors))
        return

    summary_df = build_summary_dataframe(results)
    detailed_df = build_detailed_2col_sheet(results)
    errors_df = pd.DataFrame(errors)

    inputs_info = pd.DataFrame({
        "Input item": [
            "Soil types",
            "Height values H",
            "Beta values",
            "BC/H values",
            "DE/H values",
            "EF/H values",
            "DB/H values",
            "Toe soil height h values",
            "Top width values",
            "Density types",
            "Strength states",
            "Concrete unit weight",
            "Old active pressure soils",
            "Active pressure rule for other soils",
            "Required FOS Overturning",
            "Required FOS Sliding",
            "Required FOS Bearing",
            "Bearing selection rule",
            "DE hint",
            "EF correction rule"
        ],
        "Value": [
            ", ".join(soil_types),
            ", ".join(map(str, H_values)),
            ", ".join(map(str, beta_values)),
            ", ".join(map(str, BC_coef_values)),
            ", ".join(map(str, DE_coef_values)),
            ", ".join(map(str, EF_coef_values)),
            ", ".join(map(str, DB_coef_values)),
            ", ".join(map(str, h_toe_values)),
            ", ".join(map(str, top_width_values)),
            ", ".join(density_types),
            ", ".join(strength_states),
            gamma_concrete,
            ", ".join(sorted(OLD_PA_SOILS)),
            "Use new formula with Zc",
            REQUIRED_FOS_OVERTURNING,
            REQUIRED_FOS_SLIDING,
            REQUIRED_FOS_BEARING,
            "Low density uses bearing_Low, Medium uses bearing_Medium, High uses bearing_High",
            "Suggested DE range = 0.13H to 0.23H",
            f"If EF <= top width, use EF = top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)}"
        ]
    })

    soil_properties_used = soil_db.copy()

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        inputs_info.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=0)
        soil_properties_used.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=len(inputs_info) + 3)

        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        detailed_df.to_excel(writer, sheet_name="Detailed_2Col", index=False)

        if not errors_df.empty:
            errors_df.to_excel(writer, sheet_name="Errors", index=False)

        ws_inputs = writer.sheets["Inputs_Used"]
        ws_inputs.set_column(0, 0, 32)
        ws_inputs.set_column(1, 1, 60)
        ws_inputs.freeze_panes(1, 0)

        autosize_excel_columns(writer, "Summary", summary_df)
        autosize_excel_columns(writer, "Detailed_2Col", detailed_df)

        if not errors_df.empty:
            autosize_excel_columns(writer, "Errors", errors_df)
            writer.sheets["Errors"].freeze_panes(1, 0)

        writer.sheets["Summary"].freeze_panes(1, 0)
        writer.sheets["Detailed_2Col"].freeze_panes(1, 0)

    print("\nDone.")
    print(f"Valid combinations solved: {len(results)}")
    print(f"Failed combinations: {len(errors)}")
    print(f"Excel file created: {output_file}")

    print("\nPreview of Summary sheet:")
    display(summary_df.head())


# =========================================================
# RUN
# =========================================================
main()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4056094158.py, line 268)

In [9]:
import math
import itertools
from typing import Dict, Any, List, Tuple

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# =========================================================
# UPDATED SOIL DATABASE
# =========================================================
soil_db = pd.DataFrame([
    {
        "soil_type": "CG",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 0.0,
        "phi_intermediate": 36.0, "phi_lowc_highphi": 40.0, "phi_highc_lowphi": 32.0,
        "bearing_Low": 392.4, "bearing_Medium": 588.6, "bearing_High": 784.8
    },
    {
        "soil_type": "CG PF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 29.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "CG NPF",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 30.0, "phi_lowc_highphi": 32.0, "phi_highc_lowphi": 29.0,
        "bearing_Low": 147.15, "bearing_Medium": 245.25, "bearing_High": 294.3
    },
    {
        "soil_type": "Silt",
        "gamma_bulk": 18.5, "gamma_sub": 8.69,
        "c_intermediate": 7.5, "c_lowc_highphi": 3.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 25.0, "phi_lowc_highphi": 30.0, "phi_highc_lowphi": 20.0,
        "bearing_Low": 49.05, "bearing_Medium": 147.15, "bearing_High": 294.3
    },
    {
        "soil_type": "CL",
        "gamma_bulk": 17.5, "gamma_sub": 7.69,
        "c_intermediate": 12.75, "c_lowc_highphi": 5.0, "c_highc_lowphi": 15.0,
        "phi_intermediate": 18.0, "phi_lowc_highphi": 27.0, "phi_highc_lowphi": 15.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "CH",
        "gamma_bulk": 19.0, "gamma_sub": 9.19,
        "c_intermediate": 20.0, "c_lowc_highphi": 10.0, "c_highc_lowphi": 40.0,
        "phi_intermediate": 15.0, "phi_lowc_highphi": 20.0, "phi_highc_lowphi": 12.0,
        "bearing_Low": 49.05, "bearing_Medium": 196.2, "bearing_High": 392.4
    },
    {
        "soil_type": "Boulder",
        "gamma_bulk": 20.0, "gamma_sub": 10.19,
        "c_intermediate": 0.0, "c_lowc_highphi": 0.0, "c_highc_lowphi": 12.0,
        "phi_intermediate": 60.0, "phi_lowc_highphi": 60.0, "phi_highc_lowphi": 37.0,
        "bearing_Low": 981.0, "bearing_Medium": 981.0, "bearing_High": 981.0
    },
])

DENSITY_OPTIONS = ["Low", "Medium", "High"]
STRENGTH_STATE_OPTIONS = ["Intermediate", "Low C High Phi", "High C Low Phi"]
OLD_PA_SOILS = {"CG", "CG NPF", "Boulder"}
DEFAULT_OUTPUT_FILE = "retaining_wall_fos_results.xlsx"

REQUIRED_FOS_OVERTURNING = 2.0
REQUIRED_FOS_SLIDING = 1.5
REQUIRED_FOS_BEARING = 3.0

ANGLE_RATIO_FRONT_FACE = 30.0  # 1:30


# =========================================================
# SAFE INPUT HELPERS
# =========================================================
def ask_positive_int(prompt: str) -> int:
    while True:
        raw = input(prompt).strip()
        try:
            value = int(raw)
            if value <= 0:
                print("Please enter a positive whole number.")
                continue
            return value
        except ValueError:
            print("Invalid input. Please enter a whole number like 1, 2, 3.")


def normalize_strength_state(text: str) -> str:
    t = " ".join(text.strip().split()).lower()
    mapping = {
        "intermediate": "Intermediate",
        "low c high phi": "Low C High Phi",
        "high c low phi": "High C Low Phi",
    }
    return mapping.get(t, text.title())


def ask_values(label: str, example: str, cast_func=float, allow_text=False, allowed_values=None):
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        n = ask_positive_int("How many values? : ")

        raw = input("What are they? (comma separated) : ").strip()
        if raw == "":
            print("Input cannot be empty. Please try again.")
            continue

        parts = [x.strip() for x in raw.split(",") if x.strip()]

        if len(parts) != n:
            print(f"You said {n} values, but entered {len(parts)} values. Please enter again.")
            continue

        try:
            if allow_text:
                values = [str(x).strip() for x in parts]

                if allowed_values is not None:
                    normalized = []
                    for v in values:
                        if allowed_values == STRENGTH_STATE_OPTIONS:
                            vv = normalize_strength_state(v)
                        else:
                            vv = v.title()
                        normalized.append(vv)

                    bad = [v for v in normalized if v not in allowed_values]
                    if bad:
                        print(f"These values are not allowed: {bad}")
                        print(f"Allowed values are: {allowed_values}")
                        continue
                    return normalized

                return values
            else:
                return [cast_func(x) for x in parts]

        except ValueError:
            print("Invalid input. Please enter again.")


def ask_yes_no(label: str, example: str = "Yes") -> bool:
    while True:
        print(f"\n{label}")
        print(f"Example: {example}")
        value = input("Enter Yes or No: ").strip().lower()

        if value in {"yes", "y", "true", "1"}:
            return True
        elif value in {"no", "n", "false", "0"}:
            return False
        else:
            print("Invalid input. Please enter Yes or No.")


def ask_soil_types(available_soils):
    while True:
        values = ask_values(
            label="1. Soil types",
            example="CG, CG PF, Silt, CL",
            cast_func=str,
            allow_text=True
        )

        bad = [v for v in values if v not in available_soils]
        if bad:
            print(f"These soil types are not in the database: {bad}")
            print(f"Available soil types are: {available_soils}")
            continue
        return values


def ask_output_filename(default_name: str) -> str:
    while True:
        value = input(f"\n14. Output Excel file name [default: {default_name}] : ").strip()
        if value == "":
            return default_name
        if not value.lower().endswith(".xlsx"):
            value += ".xlsx"
        return value


# =========================================================
# SOIL LOOKUP
# =========================================================
def get_phi_c_by_strength_state(row: Dict[str, Any], strength_state: str) -> Tuple[float, float]:
    if strength_state == "Intermediate":
        return row["phi_intermediate"], row["c_intermediate"]
    elif strength_state == "Low C High Phi":
        return row["phi_lowc_highphi"], row["c_lowc_highphi"]
    elif strength_state == "High C Low Phi":
        return row["phi_highc_lowphi"], row["c_highc_lowphi"]
    else:
        raise ValueError(f"Unknown strength state: {strength_state}")


def soil_lookup(soil_type: str, density_type: str, strength_state: str) -> Dict[str, Any]:
    density_type = density_type.title()
    if density_type not in DENSITY_OPTIONS:
        raise ValueError(f"density_type must be one of {DENSITY_OPTIONS}")

    if strength_state not in STRENGTH_STATE_OPTIONS:
        raise ValueError(f"strength_state must be one of {STRENGTH_STATE_OPTIONS}")

    row = soil_db.loc[soil_db["soil_type"] == soil_type]
    if row.empty:
        raise ValueError(f"Soil type '{soil_type}' not found.")

    row = row.iloc[0].to_dict()
    phi, cohesion = get_phi_c_by_strength_state(row, strength_state)

    row["density_type"] = density_type
    row["strength_state"] = strength_state
    row["phi_used"] = phi
    row["cohesion_used"] = cohesion
    row["bearing_capacity"] = row.get(f"bearing_{density_type}")
    return row


# =========================================================
# GEOTECHNICAL FUNCTIONS
# =========================================================
def rankine_Ka_Kp_sloping(beta_deg: float, phi_deg: float) -> Tuple[float, float]:
    beta = math.radians(beta_deg)
    phi = math.radians(phi_deg)

    cosb = math.cos(beta)
    term_sq = cosb**2 - math.cos(phi)**2

    if term_sq < 0:
        raise ValueError(f"Invalid case: beta={beta_deg}° is greater than allowable for phi={phi_deg}°.")

    term = math.sqrt(term_sq)

    Ka = cosb * (cosb - term) / (cosb + term)
    Kp = cosb * (cosb + term) / (cosb - term)
    return Ka, Kp


def active_pressure_old(H: float, gamma: float, c: float, Ka: float) -> float:
    Pa = 0.5 * Ka * gamma * H**2 - 2.0 * c * math.sqrt(Ka) * H
    return max(Pa, 0.0)


def active_pressure_new(H: float, gamma: float, c: float, Ka: float) -> Tuple[float, float]:
    if gamma <= 0 or Ka <= 0:
        return 0.0, 0.0

    Zc = (2.0 * c) / (gamma * math.sqrt(Ka))

    if H <= Zc:
        return 0.0, Zc

    Pa = 0.5 * (H - Zc) * (gamma * H * Ka - 2.0 * c * math.sqrt(Ka))
    return max(Pa, 0.0), Zc


def compute_case(
    soil_type: str,
    density_type: str,
    strength_state: str,
    H: float,
    beta_deg: float,
    BC_coef: float,
    DE_coef: float,
    EF_coef: float,
    DB_coef: float,
    h_toe: float,
    top_width: float,
    gamma_concrete: float = 24.0,
) -> Dict[str, Any]:

    soil = soil_lookup(soil_type, density_type, strength_state)

    gamma_bulk = soil["gamma_bulk"]
    gamma_sub = soil["gamma_sub"]
    phi = soil["phi_used"]
    cohesion = soil["cohesion_used"]
    bearing = soil["bearing_capacity"]

    delta_deg = (2.0 / 3.0) * phi

    BC = BC_coef * H
    DE = DE_coef * H
    EF_input = EF_coef * H
    DB = DB_coef * H

    angle_displacement = (H - DB) / ANGLE_RATIO_FRONT_FACE
    min_ef_required = top_width + angle_displacement

    # EXACT RULE:
    # EF_used = max(EF_input, top_width + (H-DB)/30)
    EF_used = max(EF_input, min_ef_required)

    if EF_used > EF_input:
        ef_rule_applied = "YES"
    else:
        ef_rule_applied = "NO"

    ef_check = "PASS" if EF_used >= min_ef_required else "FAIL"

    FG = BC - (DE + EF_used)

    if FG <= 0:
        raise ValueError("FG <= 0. Increase BC/H or reduce DE/H and EF/H.")

    beta_rad = math.radians(beta_deg)
    H_prime = H + math.tan(beta_rad) * FG

    Ka, Kp = rankine_Ka_Kp_sloping(beta_deg, phi)

    # Weights with EF_used
    W1 = 0.5 * (H_prime - H) * FG * gamma_bulk
    W2 = FG * (H - DB) * gamma_bulk
    W3 = top_width * (H - DB) * gamma_concrete
    W4 = 0.5 * (H - DB) * max(EF_used - top_width, 0.0) * gamma_concrete
    W5 = DB * BC * gamma_concrete
    W_total = W1 + W2 + W3 + W4 + W5

    if soil_type in OLD_PA_SOILS:
        Pa = active_pressure_old(H, gamma_bulk, cohesion, Ka)
        Zc = 0.0
        active_formula_used = "Old formula"
    else:
        Pa, Zc = active_pressure_new(H, gamma_bulk, cohesion, Ka)
        active_formula_used = "New formula with Zc"

    Pp = 0.5 * Kp * gamma_bulk * (h_toe ** 2) + 2.0 * cohesion * math.sqrt(Kp) * h_toe

    Ph = Pa * math.cos(beta_rad)
    Pv = Pa * math.sin(beta_rad)

    # Moments with EF_used
    x_W1 = DE + EF_used + 2.0 * FG / 3.0
    x_W2 = DE + EF_used + FG / 2.0
    x_W3 = DE + max(EF_used - top_width, 0.0) + top_width / 2.0
    x_W4 = DE + 2.0 * max(EF_used - top_width, 0.0) / 3.0 if W4 > 0 else 0.0
    x_W5 = BC / 2.0

    M_W1 = W1 * x_W1
    M_W2 = W2 * x_W2
    M_W3 = W3 * x_W3
    M_W4 = W4 * x_W4
    M_W5 = W5 * x_W5
    M_Pv = Pv * BC

    Mr = M_W1 + M_W2 + M_W3 + M_W4 + M_W5 + M_Pv
    Mo = Ph * H_prime / 3.0

    FS_overturning = Mr / Mo if abs(Mo) > 1e-12 else float("inf")

    delta_rad = math.radians(delta_deg)
    resisting = W_total * math.tan(delta_rad) + Pp
    driving = Ph
    FS_sliding = resisting / driving if abs(driving) > 1e-12 else float("inf")

    V = W_total + Pv
    e = (BC / 2.0) - ((Mr - Mo) / V) if abs(V) > 1e-12 else float("nan")
    pmax = (V / BC) * (1.0 + 6.0 * e / BC) if BC > 0 else float("nan")
    pmin = (V / BC) * (1.0 - 6.0 * e / BC) if BC > 0 else float("nan")

    FS_bearing = "N/A"
    if bearing is not None and not pd.isna(bearing) and max(pmax, pmin) > 0:
        FS_bearing = bearing / max(pmax, pmin)

    overturning_check = "PASS" if FS_overturning >= REQUIRED_FOS_OVERTURNING else "FAIL"
    sliding_check = "PASS" if FS_sliding >= REQUIRED_FOS_SLIDING else "FAIL"
    if FS_bearing == "N/A":
        bearing_check = "N/A"
    else:
        bearing_check = "PASS" if FS_bearing >= REQUIRED_FOS_BEARING else "FAIL"

    return {
        "Soil types": soil_type,
        "Density type": density_type,
        "Strength state": strength_state,
        "Height of structure": H,
        "inclination of soil": beta_deg,
        "Coefficient of length (BC/H)": BC_coef,
        "Coefficient of base (DE/H)": DE_coef,
        "Coefficient of EF (EF/H)": EF_coef,
        "Coefficient of height of toe (DB/H)": DB_coef,
        "Toe soil height h": h_toe,
        "Top width": top_width,

        "Length BC": BC,
        "Length DE": DE,
        "Length EF input": EF_input,
        "Length EF used": EF_used,
        "Length FG": FG,
        "Height DB": DB,

        "Angle ratio used": f"1:{int(ANGLE_RATIO_FRONT_FACE)}",
        "Angle displacement": angle_displacement,
        "Minimum EF required": min_ef_required,
        "EF rule applied": ef_rule_applied,
        "EF Check": ef_check,

        "Bulk unit weight": gamma_bulk,
        "Submerged unit weight": gamma_sub,
        "phi used": phi,
        "cohesion used": cohesion,
        "Bearing capacity": bearing,
        "delta": delta_deg,
        "Ka": Ka,
        "Kp": Kp,
        "Zc": Zc,
        "Active formula used": active_formula_used,

        "W1": W1,
        "W2": W2,
        "W3": W3,
        "W4": W4,
        "W5": W5,
        "W_total": W_total,
        "Pa": Pa,
        "Pp": Pp,
        "Ph": Ph,
        "Pv": Pv,

        "x_W1": x_W1,
        "x_W2": x_W2,
        "x_W3": x_W3,
        "x_W4": x_W4,
        "x_W5": x_W5,
        "M_W1": M_W1,
        "M_W2": M_W2,
        "M_W3": M_W3,
        "M_W4": M_W4,
        "M_W5": M_W5,
        "M_Pv": M_Pv,
        "Mr": Mr,
        "Mo": Mo,

        "eccentricity": e,
        "pmax": pmax,
        "pmin": pmin,

        "FOS_against Overturning": FS_overturning,
        "FOS_against Sliding": FS_sliding,
        "FOS_against Bearing": FS_bearing,
        "Required FOS Overturning": REQUIRED_FOS_OVERTURNING,
        "Required FOS Sliding": REQUIRED_FOS_SLIDING,
        "Required FOS Bearing": REQUIRED_FOS_BEARING,
        "Overturning Check": overturning_check,
        "Sliding Check": sliding_check,
        "Bearing Check": bearing_check,
    }


# =========================================================
# DATAFRAME BUILDERS
# =========================================================
def build_summary_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    cols = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length BC",
        "Length DE",
        "Length EF input",
        "Length EF used",
        "Length FG",
        "Height DB",
        "Angle ratio used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "phi used",
        "cohesion used",
        "Bearing capacity",
        "Ka",
        "Kp",
        "Zc",
        "Active formula used",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "x_W1",
        "x_W2",
        "x_W3",
        "x_W4",
        "x_W5",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "pmax",
        "pmin",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
        "Required FOS Overturning",
        "Required FOS Sliding",
        "Required FOS Bearing",
        "Overturning Check",
        "Sliding Check",
        "Bearing Check",
    ]
    return pd.DataFrame(results)[cols]


def build_detailed_2col_sheet(results: List[Dict[str, Any]]) -> pd.DataFrame:
    wanted_order = [
        "Soil types",
        "Density type",
        "Strength state",
        "Height of structure",
        "inclination of soil",
        "Coefficient of length (BC/H)",
        "Coefficient of base (DE/H)",
        "Coefficient of EF (EF/H)",
        "Coefficient of height of toe (DB/H)",
        "Toe soil height h",
        "Top width",
        "Length EF input",
        "Length EF used",
        "Angle displacement",
        "Minimum EF required",
        "EF rule applied",
        "EF Check",
        "W1",
        "W2",
        "W3",
        "W4",
        "W5",
        "W_total",
        "Pa",
        "Pp",
        "Ph",
        "Pv",
        "M_W1",
        "M_W2",
        "M_W3",
        "M_W4",
        "M_W5",
        "M_Pv",
        "Mr",
        "Mo",
        "eccentricity",
        "FOS_against Overturning",
        "FOS_against Sliding",
        "FOS_against Bearing",
    ]

    blocks = []
    for i, res in enumerate(results, start=1):
        block = pd.DataFrame({
            f"Parameter_{i}": wanted_order,
            f"Value_{i}": [res.get(k, "") for k in wanted_order]
        })
        blocks.append(block)

    return pd.concat(blocks, axis=1)


def autosize_excel_columns(writer, sheet_name, df):
    worksheet = writer.sheets[sheet_name]
    for idx, col in enumerate(df.columns):
        series = df[col].fillna("").astype(str) if col in df.columns else pd.Series(dtype=str)
        max_len = max([len(str(col))] + [len(x) for x in series.tolist()] + [10])
        worksheet.set_column(idx, idx, min(max_len + 2, 18 if sheet_name == "Summary" else 45))


# =========================================================
# MAIN
# =========================================================
def main():
    print("=" * 74)
    print("RETAINING WALL COMBINATION ANALYSIS")
    print("=" * 74)
    print("This script will ask the user for all values, run all combinations,")
    print("calculate FoS against overturning, sliding, and bearing,")
    print("and export the results to Excel.\n")

    print("Available soil types:")
    print(", ".join(soil_db["soil_type"].tolist()))
    print("\nAvailable density types:")
    print("Low, Medium, High")
    print("Example input: Medium, High")
    print("\nAvailable strength states:")
    print("Intermediate, Low C High Phi, High C Low Phi")
    print("Example input: Intermediate, High C Low Phi")
    print("\nDE hint:")
    print("Suggested DE range: 0.13H to 0.23H")
    print(f"\nEF rule:")
    print(f"EF_used = max(EF_input, Top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)})")

    soil_types = ask_soil_types(soil_db["soil_type"].tolist())

    H_values = ask_values(
        label="2. Height of structure H (m)",
        example="2.5, 5, 7",
        cast_func=float
    )

    beta_values = ask_values(
        label="3. Inclination of soil β (degree)",
        example="0, 5, 10, 15",
        cast_func=float
    )

    BC_coef_values = ask_values(
        label="4. Coefficient of length BC/H",
        example="0.5, 0.6, 0.7",
        cast_func=float
    )

    DE_coef_values = ask_values(
        label="5. Coefficient of base DE/H",
        example="0.13, 0.18, 0.23",
        cast_func=float
    )

    EF_coef_values = ask_values(
        label="6. Coefficient of EF (EF/H)",
        example="0.1, 0.15, 0.2",
        cast_func=float
    )

    DB_coef_values = ask_values(
        label="7. Coefficient of height of toe DB/H",
        example="0.05, 0.10, 0.15",
        cast_func=float
    )

    density_types = ask_values(
        label="8. Soil density type(s)",
        example="Medium, High",
        cast_func=str,
        allow_text=True,
        allowed_values=DENSITY_OPTIONS
    )

    strength_states = ask_values(
        label="9. Soil strength state(s)",
        example="Intermediate, High C Low Phi",
        cast_func=str,
        allow_text=True,
        allowed_values=STRENGTH_STATE_OPTIONS
    )

    h_toe_values = ask_values(
        label="10. Toe soil height h (m)",
        example="0.6, 0.7",
        cast_func=float
    )

    top_width_values = ask_values(
        label="11. Top width values (m)",
        example="0.3, 0.4, 0.5",
        cast_func=float
    )

    use_default_gamma_conc = ask_yes_no(
        label="12. Use default unit weight of concrete = 24 kN/m3 ?",
        example="Yes"
    )
    if use_default_gamma_conc:
        gamma_concrete = 24.0
    else:
        while True:
            try:
                gamma_concrete = float(input("Enter concrete unit weight (kN/m3): ").strip())
                if gamma_concrete <= 0:
                    print("Unit weight must be positive.")
                    continue
                break
            except ValueError:
                print("Invalid input. Please enter a number.")

    output_file = ask_output_filename(DEFAULT_OUTPUT_FILE)

    combos = list(itertools.product(
        soil_types,
        density_types,
        strength_states,
        H_values,
        beta_values,
        BC_coef_values,
        DE_coef_values,
        EF_coef_values,
        DB_coef_values,
        h_toe_values,
        top_width_values
    ))

    print(f"\nTotal combinations to evaluate: {len(combos)}")

    results = []
    errors = []

    for idx, (soil_type, density_type, strength_state, H, beta, BCc, DEc, EFc, DBc, htoe, topw) in enumerate(combos, start=1):
        try:
            res = compute_case(
                soil_type=soil_type,
                density_type=density_type,
                strength_state=strength_state,
                H=H,
                beta_deg=beta,
                BC_coef=BCc,
                DE_coef=DEc,
                EF_coef=EFc,
                DB_coef=DBc,
                h_toe=htoe,
                top_width=topw,
                gamma_concrete=gamma_concrete
            )
            results.append(res)
        except Exception as e:
            errors.append({
                "Combination No.": idx,
                "Soil": soil_type,
                "Density": density_type,
                "Strength State": strength_state,
                "H": H,
                "beta": beta,
                "BC/H": BCc,
                "DE/H": DEc,
                "EF/H": EFc,
                "DB/H": DBc,
                "Toe soil height h": htoe,
                "Top width": topw,
                "Error": str(e)
            })

    if not results:
        print("\nNo valid combinations were solved.")
        if errors:
            display(pd.DataFrame(errors))
        return

    summary_df = build_summary_dataframe(results)
    detailed_df = build_detailed_2col_sheet(results)
    errors_df = pd.DataFrame(errors)

    inputs_info = pd.DataFrame({
        "Input item": [
            "Soil types",
            "Height values H",
            "Beta values",
            "BC/H values",
            "DE/H values",
            "EF/H values",
            "DB/H values",
            "Toe soil height h values",
            "Top width values",
            "Density types",
            "Strength states",
            "Concrete unit weight",
            "Old active pressure soils",
            "Active pressure rule for other soils",
            "Required FOS Overturning",
            "Required FOS Sliding",
            "Required FOS Bearing",
            "Bearing selection rule",
            "DE hint",
            "EF correction rule"
        ],
        "Value": [
            ", ".join(soil_types),
            ", ".join(map(str, H_values)),
            ", ".join(map(str, beta_values)),
            ", ".join(map(str, BC_coef_values)),
            ", ".join(map(str, DE_coef_values)),
            ", ".join(map(str, EF_coef_values)),
            ", ".join(map(str, DB_coef_values)),
            ", ".join(map(str, h_toe_values)),
            ", ".join(map(str, top_width_values)),
            ", ".join(density_types),
            ", ".join(strength_states),
            gamma_concrete,
            ", ".join(sorted(OLD_PA_SOILS)),
            "Use new formula with Zc",
            REQUIRED_FOS_OVERTURNING,
            REQUIRED_FOS_SLIDING,
            REQUIRED_FOS_BEARING,
            "Low density uses bearing_Low, Medium uses bearing_Medium, High uses bearing_High",
            "Suggested DE range = 0.13H to 0.23H",
            f"EF_used = max(EF_input, Top width + (H-DB)/{int(ANGLE_RATIO_FRONT_FACE)})"
        ]
    })

    soil_properties_used = soil_db.copy()

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        inputs_info.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=0)
        soil_properties_used.to_excel(writer, sheet_name="Inputs_Used", index=False, startrow=len(inputs_info) + 3)

        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        detailed_df.to_excel(writer, sheet_name="Detailed_2Col", index=False)

        if not errors_df.empty:
            errors_df.to_excel(writer, sheet_name="Errors", index=False)

        ws_inputs = writer.sheets["Inputs_Used"]
        ws_inputs.set_column(0, 0, 32)
        ws_inputs.set_column(1, 1, 60)
        ws_inputs.freeze_panes(1, 0)

        autosize_excel_columns(writer, "Summary", summary_df)
        autosize_excel_columns(writer, "Detailed_2Col", detailed_df)

        if not errors_df.empty:
            autosize_excel_columns(writer, "Errors", errors_df)
            writer.sheets["Errors"].freeze_panes(1, 0)

        writer.sheets["Summary"].freeze_panes(1, 0)
        writer.sheets["Detailed_2Col"].freeze_panes(1, 0)

    print("\nDone.")
    print(f"Valid combinations solved: {len(results)}")
    print(f"Failed combinations: {len(errors)}")
    print(f"Excel file created: {output_file}")

    print("\nPreview of Summary sheet:")
    display(summary_df.head())


# =========================================================
# RUN
# =========================================================
main()

RETAINING WALL COMBINATION ANALYSIS
This script will ask the user for all values, run all combinations,
calculate FoS against overturning, sliding, and bearing,
and export the results to Excel.

Available soil types:
CG, CG PF, CG NPF, Silt, CL, CH, Boulder

Available density types:
Low, Medium, High
Example input: Medium, High

Available strength states:
Intermediate, Low C High Phi, High C Low Phi
Example input: Intermediate, High C Low Phi

DE hint:
Suggested DE range: 0.13H to 0.23H

EF rule:
EF_used = max(EF_input, Top width + (H-DB)/30)

1. Soil types
Example: CG, CG PF, Silt, CL


How many values? :  7
What are they? (comma separated) :  CG, CG PF, CG NPF, Silt, CL, CH, Boulder



2. Height of structure H (m)
Example: 2.5, 5, 7


How many values? :  5
What are they? (comma separated) :  2, 3, 5, 7, 9



3. Inclination of soil β (degree)
Example: 0, 5, 10, 15


...

How many values? :  1
What are they? (comma separated) :  .3



12. Use default unit weight of concrete = 24 kN/m3 ?
Example: Yes


Enter Yes or No:  Yes

14. Output Excel file name [default: retaining_wall_fos_results.xlsx] :  All final 3



Total combinations to evaluate: 117600


ValueError: This sheet is too large! Your sheet size is: 39, 210064 Max sheet size is: 1048576, 16384